# Clinician Adoption Personas: a multi-agent pipeline

This notebook implements the adoption-persona framework (Alrefaei et al.) with groups of Claude agents. **The researcher defines the adoption attributes**; the agents then extract, rate, cluster and build personas from them.

**Every step uses at least 3 independent agents that verify each other** (`N_AGENTS = 3`). Each agent works without seeing the others' output. Their results are then combined by majority vote or reconciled by a lead agent, and the notebook reports how well they agreed.

| # | Agent group | Independent agents | How they verify each other |
|---|---|---|---|
| 1 | **Familiarization & speaker identification** | 3 *Readers* per transcript + 1 *Lead* | Majority vote on who is interviewer / participant and on the participant's role (physician / nurse). The Lead resolves every disagreement. |
| 2 | **Thematic analysis** | 3 *Coders* per transcript → 3 *Theme Developers* → 1 *Reconciler* → 3 *Theme Verifiers* | Themes are developed independently and reconciled (recording how many developers found each one). A theme is kept only if a majority of verifiers confirm it; role flags are set by majority. |
| 3 | **Statement extraction & rating** | 3 *Extractors* + 3 *Raters* per participant (+ *Adjudicator*) | A statement is kept only if at least 2 of the 3 extractors found it for the same construct (and it is verbatim). The final rating is the median of 3 independent ratings; a spread of 2+ points goes to the Adjudicator. Inter-rater agreement is reported. |
| 4 | **Cluster analysis** | 3 *Cluster Analysts* per role + 3 *Profile Matchers* + 1 *Reconciler* | Cut points by majority vote (**you confirm**). Physician and nurse clusters are merged into one profile only when a majority of matchers agree. |
| 5 | **Persona cards** | 1 *Designer* + 3 *Verifiers* per profile | Evidence, profile-consistency and bias / actionability verifiers check each persona; the Designer revises if any verifier raises issues. |
| 6 | **Living personas** | 1 *Persona* + 2 *Checkers* per answer | A grounding checker and a character checker review every answer you get from a persona. |

Optional extras from the paper: **RoBERTa sentiment features** (CardiffNLP) and a **post hoc correlation** of SUS / perceived ease of use with the attributes. The clustering, sentiment model and statistics are Python, not agents.

### How to run in Google Colab
1. Click the 🔑 **Secrets** icon. Add `ANTHROPIC_API_KEY` and turn on **Notebook access**.
2. Review the **researcher input** cells: study settings (Step 2) and attributes (Step 3).
3. **Runtime → Run all.** The run pauses at Step 9 so you can confirm the cluster cut points.

The built-in sample transcripts are short and fictional. They only demonstrate the flow.

## Step 1: Install

In [ ]:
!pip install -q anthropic python-docx scikit-learn transformers

## Step 2: Study settings and connection to Claude

In [ ]:
import anthropic, json, os, re, threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    API_KEY = os.environ["ANTHROPIC_API_KEY"]
client = anthropic.Anthropic(api_key=API_KEY)

MODEL = "claude-opus-5-5"
N_AGENTS = 3                     # independent agents per task that verify each other (never fewer than 3)
MAX_PARALLEL = 4                 # API calls running at the same time; lower it if you hit rate limits
OUTPUT_DIR = "persona_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- RESEARCHER INPUT ----------------
RESEARCH_CONTEXT = """
System: an AI-enabled clinical decision support system (CDSS) for chronic pain assessment.
It estimates aspects of the patient's pain experience from eye-movement measures and presents
them in a clinical dashboard. Data: think-aloud sessions while clinicians used the dashboard
on a representative chronic pain case, semistructured interviews (informed by Innovation
Diffusion Theory and the Technology Acceptance Model), and clinicians' explanations of their
SUS and perceived ease-of-use ratings.
Goal: develop clinician adoption personas to inform design and implementation.
"""
USER_GROUPS = ["Physician", "Nurse"]     # the professional roles clustered separately
USE_ROBERTA_SENTIMENT = True             # add RoBERTa sentiment scores as separate clustering features
# ---------------------------------------------------

N = max(3, N_AGENTS)
_api_slots = threading.Semaphore(MAX_PARALLEL)


def run_agent(role: str, task: str, output_model=None, effort: str = "medium"):
    """Run one agent. role = its system prompt. With output_model (a Pydantic class),
    the answer comes back as a validated object instead of text."""
    kwargs = dict(
        model=MODEL, max_tokens=64000, system=role,
        messages=[{"role": "user", "content": task}],
        output_config={"effort": effort},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",                     # if Claude declines, retry on a fallback model
    )
    if output_model is not None:
        kwargs["output_format"] = output_model
    with _api_slots:
        with client.beta.messages.stream(**kwargs) as stream:
            msg = stream.get_final_message()
    if msg.stop_reason == "refusal":
        raise RuntimeError(f"Request declined: {msg.stop_details}")
    if msg.stop_reason == "max_tokens":
        raise RuntimeError("Answer was cut off (max_tokens).")
    if output_model is not None:
        return msg.parsed_output
    return "".join(b.text for b in msg.content if b.type == "text")

def parallel(fn, items):
    items = list(items)
    if not items:
        return []
    with ThreadPoolExecutor(max_workers=min(32, len(items))) as pool:   # API calls are capped by MAX_PARALLEL
        return list(pool.map(fn, items))

def independent(title, i, prompt):
    return (f"You are {title} {i} of {N}, working independently. Other agents do the same task separately and "
            f"your work will be cross-checked against theirs, so rely only on your own careful reading.\n\n{prompt}")

def ensemble(title, prompt, task, output_model, effort="medium"):
    """Run N independent agents on the same task and return all N answers."""
    return parallel(lambda i: run_agent(independent(title, i, prompt), task, output_model, effort), range(1, N + 1))

def majority(values):
    """Most common value, how many agents chose it, and the full vote count."""
    votes = Counter(values)
    top = max(votes.values())
    tied = sorted(v for v, c in votes.items() if c == top)
    return tied[len(tied) // 2], top, dict(votes)

def save_json(name, obj):
    with open(os.path.join(OUTPUT_DIR, name), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

print(run_agent("You are a helpful assistant.", "Reply with the single word: ready", effort="low"))

## Step 3: Researcher input, the adoption attributes

**This step is the researcher's job, not the agents'.** Define each construct, its theoretical source, and what a low (1) and high (5) rating means. The agents extract statements and rate them only against these definitions. Edit the wording freely, and add or remove attributes. Everything downstream adapts.

In [ ]:
ATTRIBUTES = {
    "fit_with_workflow": {
        "name": "Fit with clinical workflow",
        "source": "Innovation Diffusion Theory (compatibility); sociotechnical perspective (Sittig & Singh, 2010)",
        "definition": "The degree to which the clinician perceives the system as compatible with their existing "
                      "clinical workflow, routines, time constraints and professional responsibilities.",
        "low": "The system would disrupt the workflow, add steps or time, or not fit their responsibilities.",
        "high": "The system fits naturally into existing routines and responsibilities without extra burden.",
    },
    "usability": {
        "name": "Usability",
        "source": "System Usability Scale (Brooke, 1996)",
        "definition": "The clinician's perception of how well the dashboard supports effective, efficient and "
                      "satisfying use: layout, clarity, organization and presentation of information.",
        "low": "The interface is confusing, cluttered, inconsistent or hard to interpret.",
        "high": "The interface is clear, well organized and pleasant to use.",
    },
    "trust_in_ai": {
        "name": "Trust in AI",
        "source": "Trust in AI-based CDSS (Schwartz et al., 2022; Tun et al., 2025)",
        "definition": "The clinician's confidence in the credibility, accuracy, reliability and transparency of "
                      "the AI-generated, eye-movement-based pain information.",
        "low": "Doubts the validity or accuracy of the AI output; would not rely on it.",
        "high": "Considers the AI output credible and reliable enough to inform clinical decisions.",
    },
    "relative_advantage": {
        "name": "Relative advantage",
        "source": "Innovation Diffusion Theory (Rogers, 2003)",
        "definition": "The degree to which the clinician perceives the system as better than current practice "
                      "for chronic pain assessment (e.g., adds objective information, improves decisions or care).",
        "low": "Sees no added value over current practice and self-report measures.",
        "high": "Sees clear, substantial clinical value over current practice.",
    },
    "ease_of_use": {
        "name": "Ease of use",
        "source": "Technology Acceptance Model (Davis, 1989)",
        "definition": "The degree to which the clinician believes using the system would be free of effort: "
                      "easy to learn, operate and interpret.",
        "low": "Using or learning the system would require substantial effort or training.",
        "high": "The system would be effortless to learn and use.",
    },
    "intention_to_use": {
        "name": "Intention to use",
        "source": "Technology Acceptance Model (Davis, 1989)",
        "definition": "The clinician's stated willingness or plan to use the system in their clinical practice.",
        "low": "Would not use the system.",
        "high": "Clearly intends to use the system in practice.",
    },
}
ATTRIBUTE_KEYS = list(ATTRIBUTES)
SHORT = {k: v["name"] for k, v in ATTRIBUTES.items()}

def attribute_brief(keys=ATTRIBUTE_KEYS):
    return "\n\n".join(
        f"[{k}] {ATTRIBUTES[k]['name']} ({ATTRIBUTES[k]['source']})\n"
        f"Definition: {ATTRIBUTES[k]['definition']}\n1 (low): {ATTRIBUTES[k]['low']}\n5 (high): {ATTRIBUTES[k]['high']}"
        for k in keys)
print(f"{len(ATTRIBUTES)} attributes:", ", ".join(SHORT.values()))

## Step 4: Load the transcripts

Use one file per participant (`.txt` or `.docx`). Speaker labels can be anything. Group 1 works out who is who.

In [ ]:
USE_SAMPLE_DATA = True

SAMPLE_TRANSCRIPTS = {
"C01.txt": """Interviewer: Please go through the dashboard as you would in a consultation.
Speaker 2: OK, so this is the pain estimate from the eye movements. I like that it sits next to the medication list and the pain history. That's what I'd look at anyway.
Interviewer: What do you think of the information?
Speaker 2: For chronic pain this is something we really lack, an objective measure. If it's validated it would help me a lot in deciding whether a treatment is working.
Interviewer: Would you trust it?
Speaker 2: I'd want to know how the estimate is calculated, but if there's an explanation I'd trust it alongside my own judgment.
Interviewer: Would you use it?
Speaker 2: Yes, definitely, I'd use it for my follow-up patients. Adding the eye-tracking step to the visit might take time though, who would do it?
Interviewer: You gave the system a high usability score. Why?
Speaker 2: It's clean, I found everything quickly, very easy to use.""",

"C02.txt": """Speaker A: I'm an anesthesiologist, I've done pain medicine for twenty years.
Speaker B: Thanks. Please think aloud as you use the dashboard.
Speaker A: Honestly I don't see what this adds. Pain is what the patient tells me. An eye-movement number won't change my treatment.
Speaker B: How about trusting the estimate?
Speaker A: I don't trust it. It's a black box, and I haven't seen the evidence.
Speaker B: Would you use it?
Speaker A: No. Not unless there are large trials showing benefit. The screen is easy enough, that's not the issue, it's that it slows my clinic down.""",

"C03.txt": """Interviewer: Can you walk me through the case using the dashboard?
Dr. Reem: Sure. Wow, the longitudinal trend is great, I can see the pain trajectory across visits. I'd love opioid alerts here too.
Interviewer: How does it compare to what you do now?
Dr. Reem: Much better than a 0 to 10 scale alone. It gives me something objective to discuss with the patient.
Interviewer: Trust?
Dr. Reem: I trust it as one input. I'd use it tomorrow if I could.
Interviewer: Fit with your workflow?
Dr. Reem: It would need to be in the EHR, otherwise it's another login. But I'd make it work.
Interviewer: Ease of use?
Dr. Reem: Very intuitive, I didn't need any help.""",

"C04.txt": """Interviewer: Please think aloud as you look at the dashboard.
Speaker 2: There's a lot here. I'm not sure what this AI percentage means. I'd prefer familiar terms.
Interviewer: Would it help your practice?
Speaker 2: It could, maybe, for patients who struggle to describe pain. But I'd need to see it work in our clinic first.
Interviewer: Do you trust the estimate?
Speaker 2: Somewhat. I'd want to compare it with what the patient says for a while.
Interviewer: Would you use it?
Speaker 2: Possibly, if it doesn't add work. Our visits are fifteen minutes. A simpler version with just the essentials would help.""",

"C05.txt": """Interviewer: Tell me what you see as you use the dashboard.
Speaker 2: The layout is clear, the pain estimate is easy to find. I'd want the vitals and history next to it to interpret it.
Interviewer: How useful is it compared to now?
Speaker 2: Useful for monitoring treatment response, it adds a piece we don't have.
Interviewer: Trust in the information?
Speaker 2: I'd trust it if it explained why it gives that estimate. Right now it doesn't tell me why.
Interviewer: Would you use it?
Speaker 2: Yes, I think I would, especially if it fits into the EHR and doesn't add steps.""",

"N01.txt": """Interviewer: You're a registered nurse in a pain clinic. Please think aloud using the dashboard.
Speaker 2: OK. So I'd be the one collecting the eye-movement data at intake? That's a new task for us, and intake is already busy.
Interviewer: What do you think of the information?
Speaker 2: It would help me flag patients for the physician. That's valuable, it supports what patients tell us.
Interviewer: Trust?
Speaker 2: I'm not sure yet. I'd need training on what the numbers mean.
Interviewer: Would you use it?
Speaker 2: If the clinic adopts it, yes, but I'd like a simpler nurse view for documentation.""",

"N02.txt": """Speaker 1: Can you introduce yourself?
Speaker 2: I'm an oncology nurse, about eight years.
Speaker 1: Please use the dashboard and think aloud.
Speaker 2: This is exciting! Patients often under-report pain, so having an objective measure is a big deal. I trust it as long as it's validated.
Speaker 1: Ease of use?
Speaker 2: Very easy, the screen is clear and I'd learn it in minutes.
Speaker 1: Would you use it?
Speaker 2: Absolutely, I'd use it with every chronic pain patient and share it with the doctors.""",

"N03.txt": """Interviewer: Please walk me through the dashboard.
Speaker 2: Hmm, there are many sections. I'm a community health nurse, I don't usually see this kind of data.
Interviewer: Is it useful compared to now?
Speaker 2: It could add something, but I'm cautious. I'd want to see it working before relying on it.
Interviewer: Trust?
Speaker 2: Not fully. How accurate is the eye tracking?
Interviewer: Would you use it?
Speaker 2: Maybe, if it fits our home visits and doesn't need a lot of training. A short version would be better for us.""",
}

def read_file(name, data):
    if name.lower().endswith(".docx"):
        import docx, io
        return "\n".join(p.text for p in docx.Document(io.BytesIO(data)).paragraphs)
    return data.decode("utf-8", errors="ignore")

if USE_SAMPLE_DATA:
    raw_files = SAMPLE_TRANSCRIPTS
else:
    from google.colab import files
    raw_files = {n: read_file(n, d) for n, d in files.upload().items()}

transcripts = [{"participant_id": os.path.splitext(name)[0], "file": name, "raw_text": text}
               for name, text in sorted(raw_files.items())]
for t in transcripts:
    print(t["participant_id"], f"{len(t['raw_text']):,} chars")

## Step 5: Output schemas

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

AttributeKey = Literal[tuple(ATTRIBUTE_KEYS)]

class SpeakerRole(BaseModel):
    speaker_label: str = Field(description="Label exactly as written in the transcript, e.g. 'Speaker 2', 'Dr. Reem'")
    role: Literal["interviewer", "participant", "other"]
    reasoning: str

class Familiarization(BaseModel):
    speakers: list[SpeakerRole]
    user_group: str = Field(description="The participant's professional role, chosen from the given user groups")
    user_group_evidence: str
    profile: str = Field(description="Specialty, experience and other details the participant states; 'not stated' otherwise")
    summary: str
    initial_observations: list[str]

class RoleAssignment(BaseModel):
    participant_id: str
    user_group: str

class DatasetOverview(BaseModel):
    overview: str
    assignments: list[RoleAssignment]
    data_quality_notes: list[str]

class Code(BaseModel):
    code: str
    description: str
    quote: str

class CodingResult(BaseModel):
    codes: list[Code]

class Theme(BaseModel):
    theme: str
    definition: str
    sub_themes: list[str]
    participant_ids: list[str]
    groups_present: list[str]
    group_flag: Literal["shared across groups", "group-specific", "divergent across groups"]
    group_differences: str
    related_attributes: list[str]
    key_quotes: list[str]

class ThematicMap(BaseModel):
    themes: list[Theme]
    cross_group_summary: str

class ExtractedStatement(BaseModel):
    attribute: AttributeKey
    statement: str = Field(description="The participant's words copied verbatim")
    why_it_fits: str

class Extraction(BaseModel):
    statements: list[ExtractedStatement]
    attributes_without_evidence: list[str]

class StatementRating(BaseModel):
    statement_id: int
    rating: int = Field(description="1 (low level of the attribute) to 5 (high level)")
    rationale: str

class RatingSheet(BaseModel):
    ratings: list[StatementRating]

class CutRecommendation(BaseModel):
    recommended_k: int
    coherence_by_k: list[str] = Field(description="For each candidate k, whether each cluster is qualitatively coherent")
    rationale: str

class Profile(BaseModel):
    profile_name: str = Field(description="Short adoption profile name, e.g. 'Cautious Evaluator'")
    clusters: list[str] = Field(description="Cluster IDs merged into this profile, e.g. 'Physician-2'")
    adoption_pattern: str
    role_differences: str

class ProfileMatching(BaseModel):
    profiles: list[Profile]
    rationale: str

class RoleNeeds(BaseModel):
    role: str
    needs: list[str]

class PersonaCard(BaseModel):
    persona_name: str
    profile_name: str
    tagline_quote: str = Field(description="A verbatim participant quote")
    description: str = Field(description="3-4 sentences describing how this profile evaluates and adopts the system")
    goals: list[str]
    adoption_drivers: list[str]
    adoption_concerns: list[str]
    conditions_for_use: list[str]
    role_specific_needs: list[RoleNeeds]
    design_requirements: list[str]
    implementation_strategies: list[str]
    supporting_quotes: list[str] = Field(description="2-4 verbatim participant quotes")

# ---- verification and consensus ----
class SpeakerResolution(BaseModel):
    participant_id: str
    speaker_label: str
    role: Literal["interviewer", "participant", "other"]

class LeadDecision(BaseModel):
    overview: str
    assignments: list[RoleAssignment]
    speaker_resolutions: list[SpeakerResolution] = Field(description="Final role for every DISPUTED speaker label")
    data_quality_notes: list[str]

class ReconciledTheme(Theme):
    found_by: list[int] = Field(description="Which theme developers (1..N) identified this theme")

class ReconciledMap(BaseModel):
    themes: list[ReconciledTheme]
    cross_group_summary: str

class ThemeVerdict(BaseModel):
    theme_id: int
    supported: bool
    correct_flag: Literal["shared across groups", "group-specific", "divergent across groups"]
    comment: str

class ThemeVerification(BaseModel):
    verdicts: list[ThemeVerdict]

class ProfileNaming(BaseModel):
    profiles: list[Profile]

class PersonaVerification(BaseModel):
    approved: bool
    issues: list[str]

class ReplyCheck(BaseModel):
    verdict: Literal["pass", "minor issues", "fail"]
    issues: list[str]

## Step 6: Agent group 1, familiarization and speaker identification (3 readers per transcript)

In [ ]:
READER = f"""You are a qualitative researcher in the familiarization phase of thematic analysis.
Read the whole transcript (think-aloud, interview and questionnaire explanations).
1. Identify every speaker label: who is the interviewer (asks questions, gives instructions) and who is
   the participant (describes their own experience). Copy labels exactly as written.
2. Determine the participant's professional role, choosing from: {USER_GROUPS}. Quote your evidence.
   If the role is not stated, infer cautiously and say so.
3. Summarize the session and note initial observations. Never invent details.
Research context: {RESEARCH_CONTEXT}"""

FAMILIARIZATION_LEAD = f"""You lead the familiarization phase. {N} independent readers read every transcript.
For each participant you receive the majority decisions and any disagreements between the readers.
Resolve every disagreement by checking the transcript: give the final role for each disputed speaker label,
and confirm every participant's professional role (one of {USER_GROUPS}). Note data-quality issues."""

def relabel(text, speakers, pid):
    """Rewrite speaker labels as INTERVIEWER / PARTICIPANT so later agents never mix them up."""
    for label in sorted(speakers, key=len, reverse=True):
        new = {"interviewer": "INTERVIEWER", "participant": f"PARTICIPANT {pid}"}.get(speakers[label], "OTHER")
        text = re.sub(rf"(?im)^(\s*(?:\[[^\]]*\]\s*)?){re.escape(label)}(\s*(?:\([^)]*\))?\s*:)",
                      rf"\g<1>{new}\g<2>", text)
    return text

def read_transcript(t):
    readers = ensemble("Reader", READER, f"Participant ID: {t['participant_id']}\n\nTranscript:\n{t['raw_text']}",
                       Familiarization)
    votes = {}
    for r in readers:
        for s in r.speakers:
            votes.setdefault(s.speaker_label.strip(), []).append(s.role)
    speakers, disputes = {}, []
    for label, v in votes.items():
        speakers[label], n, counts = majority(v)
        if n < len(readers):                     # readers disagreed, or some readers missed this label
            disputes.append(f"speaker '{label}': {counts} ({len(readers) - len(v)} reader(s) did not list it)")
    group, n, counts = majority([r.user_group for r in readers])
    if n < len(readers):
        disputes.append(f"professional role: {counts}")
    print(f"  {t['participant_id']}: {N} readers, {'unanimous' if not disputes else str(len(disputes)) + ' disagreement(s)'}")
    return {"readers": readers, "speakers": speakers, "user_group": group, "disputes": disputes}

print(f"Group 1: {N} readers per transcript are working...")
for t, res in zip(transcripts, parallel(read_transcript, transcripts)):
    t.update(res)
    t["familiarization"] = res["readers"][0]

lead_input = "\n\n".join(
    f"## {t['participant_id']}\nMajority speaker roles: {t['speakers']}\nMajority professional role: {t['user_group']}\n"
    f"Role evidence from readers: {[r.user_group_evidence for r in t['readers']]}\n"
    f"Disagreements: {t['disputes'] or 'none'}"
    + (f"\nTranscript (for resolving disagreements):\n{t['raw_text'][:4000]}" if t["disputes"] else "")
    for t in transcripts)
lead = run_agent(FAMILIARIZATION_LEAD, lead_input, LeadDecision, effort="high")
confirmed = {a.participant_id: a.user_group for a in lead.assignments}
for t in transcripts:
    t["user_group"] = confirmed.get(t["participant_id"], t["user_group"])
for r in lead.speaker_resolutions:
    for t in transcripts:
        if t["participant_id"] == r.participant_id and r.speaker_label.strip() in t["speakers"]:
            t["speakers"][r.speaker_label.strip()] = r.role
for t in transcripts:
    t["clean_text"] = relabel(t["raw_text"], t["speakers"], t["participant_id"])

by_pid = {t["participant_id"]: t for t in transcripts}
unanimous = sum(not t["disputes"] for t in transcripts)
print(f"\nReaders unanimous on {unanimous}/{len(transcripts)} transcripts"
      + ("; the Lead resolved the disagreements." if unanimous < len(transcripts) else "."))
for t in transcripts:
    print(f"{t['participant_id']:>5} [{t['user_group']}]  " + ", ".join(f"{l} = {r}" for l, r in t["speakers"].items()))
print("\nParticipants per role:", {g: sum(t["user_group"] == g for t in transcripts) for g in USER_GROUPS})
for note in lead.data_quality_notes:
    print("  ⚠", note)
save_json("1_familiarization.json", {"lead": lead.model_dump(), "participants": [
    {"participant_id": t["participant_id"], "user_group": t["user_group"], "speakers": t["speakers"],
     "disagreements": t["disputes"], "readers": [r.model_dump() for r in t["readers"]],
     "clean_transcript": t["clean_text"]} for t in transcripts]})

## Step 7: Agent group 2, thematic analysis with role differences flagged (3 coders, 3 developers, 3 verifiers)

In [ ]:
CODER = f"""You are a qualitative coder (Braun & Clarke, phase 2). Code ONLY the PARTICIPANT's words.
Create concise codes about the clinician's experience with the system, their evaluation of it and their
adoption perceptions. Every code needs a short verbatim quote. Research context: {RESEARCH_CONTEXT}"""

THEME_DEVELOPER = f"""You are a senior qualitative researcher (Braun & Clarke, phases 3-5). Organize all codes
into themes. For each theme record the participants and roles it appears in, and FLAG it:
"shared across groups", "group-specific" (only or mainly one role) or "divergent across groups"
(roles talk about it in contrasting ways). Also list which adoption attributes it relates to:
{list(SHORT.values())}. Research context: {RESEARCH_CONTEXT}"""

THEME_RECONCILER = f"""You reconcile {N} thematic maps that were developed independently from the same codes.
Merge themes that describe the same pattern and keep distinct ones. In found_by, list which developers (1-{N})
identified each theme. Keep a theme found by only one developer only if the codes clearly support it.
Record participants, roles, the role flag and related attributes. Research context: {RESEARCH_CONTEXT}"""

THEME_VERIFIER = """You verify a reconciled thematic map against the codes (Braun & Clarke, phase 4: reviewing themes).
For EVERY theme (identified by theme_id), decide whether the codes support it (grounded, coherent and distinct
from other themes) and what the correct role flag is. Comment briefly."""

def code_transcript(t):
    coders = ensemble("Coder", CODER, f"Role: {t['user_group']}\n\n{t['clean_text']}", CodingResult)
    print(f"  {t['participant_id']}: {N} coders finished ({sum(len(c.codes) for c in coders)} codes)")
    return [(i, c) for i, res in enumerate(coders, 1) for c in res.codes]

print(f"Group 2: {N} coders per transcript are working...")
for t, codes in zip(transcripts, parallel(code_transcript, transcripts)):
    t["codes"] = codes
all_codes = "\n\n".join(f"## {t['participant_id']} ({t['user_group']})\n" +
                         "\n".join(f"- [coder {i}] {c.code}: {c.description} | \"{c.quote}\"" for i, c in t["codes"])
                         for t in transcripts)

print(f"{N} theme developers are working independently...")
maps = ensemble("Theme developer", THEME_DEVELOPER, f"Codes:\n{all_codes}", ThematicMap, effort="high")
print("Reconciler is merging the thematic maps...")
reconciled = run_agent(THEME_RECONCILER, f"Codes:\n{all_codes}\n\n" + "\n\n".join(
    f"## Thematic map from developer {i}\n{m.model_dump_json(indent=1)}" for i, m in enumerate(maps, 1)),
    ReconciledMap, effort="high")
print(f"{N} theme verifiers are checking...")
numbered = "\n\n".join(f"[theme_id {i}] {th.model_dump_json(indent=1)}" for i, th in enumerate(reconciled.themes))
verifications = ensemble("Theme verifier", THEME_VERIFIER, f"Codes:\n{all_codes}\n\nThemes:\n{numbered}",
                         ThemeVerification)

final_themes, theme_audit = [], []
for i, th in enumerate(reconciled.themes):
    verdicts = [v for ver in verifications for v in ver.verdicts if v.theme_id == i]
    support = sum(v.supported for v in verdicts)
    flag, n_flag, _ = majority([v.correct_flag for v in verdicts]) if verdicts else (th.group_flag, 0, {})
    keep = support > N / 2
    if keep:
        if n_flag > N / 2:
            th.group_flag = flag
        final_themes.append(Theme(**th.model_dump(exclude={"found_by"})))
    theme_audit.append({"theme": th.theme, "found_by_developers": len(set(th.found_by)), "verified_by": support,
                        "kept": keep, "flag": th.group_flag, "comments": [v.comment for v in verdicts]})
thematic_map = ThematicMap(themes=final_themes, cross_group_summary=reconciled.cross_group_summary)

import pandas as pd
from IPython.display import display, Markdown
display(pd.DataFrame(theme_audit)[["theme", "found_by_developers", "verified_by", "kept", "flag"]]
        .rename(columns={"found_by_developers": f"found by (of {N})", "verified_by": f"verified by (of {N})"}))
for th in thematic_map.themes:
    print(f"\n■ {th.theme}  [{th.group_flag}]  roles: {', '.join(th.groups_present)}")
    print(f"  {th.definition}")
    if th.group_differences.strip().lower() not in ("", "none"):
        print(f"  Role differences: {th.group_differences}")
theme_brief = "\n".join(f"- {th.theme} [{th.group_flag}]: {th.definition}" for th in thematic_map.themes)
save_json("2_thematic_analysis.json", {
    "codes": {t["participant_id"]: [{"coder": i, **c.model_dump()} for i, c in t["codes"]] for t in transcripts},
    "developer_maps": [m.model_dump() for m in maps], "reconciled": reconciled.model_dump(),
    "verification": theme_audit, "thematic_map": thematic_map.model_dump()})

## Step 8: Agent group 3, extract statements and rate them 1–5 (3 extractors, 3 raters)

For each participant:
1. **3 Extractors** independently copy every participant statement that fits a construct definition. A statement is **kept only if at least 2 of the 3 extractors found it for the same construct** and it appears word for word in the transcript.
2. **3 Raters** each rate every kept statement 1–5 on its construct, without seeing each other's ratings. They judge both the qualitative meaning and the sentiment of the statement.
3. The final rating is the **median** of the 3 ratings. If the ratings spread by 2 or more points, the **Adjudicator** decides.
4. Statement ratings are averaged into a **participant-level score per attribute**, as in the paper. Agreement between extractors and raters is reported.

In [ ]:
from difflib import SequenceMatcher
from itertools import combinations
from statistics import median
import numpy as np
from sklearn.metrics import cohen_kappa_score

MIN_EXTRACTOR_AGREEMENT = N // 2 + 1    # 2 of 3: a statement must be found by a majority of extractors

EXTRACTOR = f"""You extract evidence for predefined adoption constructs from a clinician's session.
Copy, VERBATIM, every statement by the PARTICIPANT (never the interviewer) that fits one of these
construct definitions. A statement may fit more than one construct; list it once per construct.
Use only the definitions below; do not invent new constructs. List constructs with no evidence.

{attribute_brief()}

Research context: {RESEARCH_CONTEXT}"""

RATER = f"""You are a rater in a qualitative study of technology adoption.
Rate each statement on its assigned construct from 1 (low level of the construct) to 5 (high level),
using only the definitions and anchors below. Consider both what the clinician says and the sentiment
of how they say it (enthusiastic, hesitant, conditional, negative). 3 = neutral, mixed or conditional.
Rate each statement on its own merits.

{attribute_brief()}"""

ADJUDICATOR = f"""You are the senior adjudicator. {N} independent raters disagreed by 2+ points on these
statements. Read each statement, its construct definition and all rationales, then give the final 1-5
rating with a rationale.

{attribute_brief()}"""

def _norm(s):
    return re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", s.lower())).strip()

def verbatim(quote, text):
    q = _norm(quote.strip('"“”'))
    return bool(q) and q in _norm(text)

def same_statement(a, b):
    a, b = _norm(a), _norm(b)
    return bool(a and b) and (a in b or b in a or SequenceMatcher(None, a, b).ratio() >= 0.8)

def consensus_statements(extractions, text):
    groups = []
    for i, ext in enumerate(extractions, 1):
        for s in ext.statements:
            if s.attribute not in ATTRIBUTES:
                continue
            for g in groups:
                if g["attribute"] == s.attribute and same_statement(g["versions"][0], s.statement):
                    g["versions"].append(s.statement); g["extractors"].add(i)
                    break
            else:
                groups.append({"attribute": s.attribute, "versions": [s.statement], "extractors": {i}})
    kept = []
    for g in groups:
        valid = [v for v in g["versions"] if verbatim(v, text)]
        if len(g["extractors"]) >= MIN_EXTRACTOR_AGREEMENT and valid:
            kept.append({"attribute": g["attribute"], "statement": max(valid, key=len), "extractors": len(g["extractors"])})
    return kept, len(groups)

def rate_participant(t):
    extractions = ensemble("Extractor", EXTRACTOR, f"Participant {t['participant_id']} ({t['user_group']})\n\n"
                           f"{t['clean_text']}", Extraction, effort="high")
    stmts, n_candidates = consensus_statements(extractions, t["raw_text"])
    if not stmts:
        print(f"  {t['participant_id']}: no statements reached extractor agreement")
        return []
    sheet = "\n".join(f"[{i}] construct={s['attribute']}: \"{s['statement']}\"" for i, s in enumerate(stmts))
    sheets = ensemble("Rater", RATER, f"Participant {t['participant_id']} ({t['user_group']}). Statements to rate:\n{sheet}",
                      RatingSheet)
    by_rater = [{r.statement_id: r for r in sh.ratings} for sh in sheets]

    rows = []
    for i, s in enumerate(stmts):
        row = {"participant_id": t["participant_id"], "user_group": t["user_group"], "statement_id": i, **s}
        for j, br in enumerate(by_rater, 1):
            row[f"rater_{j}"] = min(max(br[i].rating, 1), 5) if i in br else None
            row[f"rationale_{j}"] = br[i].rationale if i in br else ""
        given = [row[f"rater_{j}"] for j in range(1, N + 1) if row[f"rater_{j}"] is not None]
        row["spread"] = max(given) - min(given) if given else None
        row["rating"] = median(given) if given else None
        rows.append(row)

    disputed = [r for r in rows if r["spread"] is not None and r["spread"] >= 2]
    if disputed:
        adj = run_agent(ADJUDICATOR, "\n\n".join(
            f"[{r['statement_id']}] construct={r['attribute']}: \"{r['statement']}\"\n" + "\n".join(
                f"Rater {j} = {r[f'rater_{j}']} ({r[f'rationale_{j}']})" for j in range(1, N + 1))
            for r in disputed), RatingSheet)
        final = {a.statement_id: min(max(a.rating, 1), 5) for a in adj.ratings}
        for r in rows:
            if r["statement_id"] in final:
                r["rating"] = final[r["statement_id"]]
                r["adjudicated"] = True
    for r in rows:
        r.setdefault("adjudicated", False)
    print(f"  {t['participant_id']}: {len(rows)} of {n_candidates} candidate statements reached extractor agreement, "
          f"{len(disputed)} adjudicated")
    return rows

print(f"Group 3: {N} extractors and {N} raters per participant are working...")
statements = pd.DataFrame([r for rows in parallel(rate_participant, transcripts) for r in rows])

rater_cols = [f"rater_{j}" for j in range(1, N + 1)]
kappas = []
for a, b in combinations(rater_cols, 2):
    pair = statements[[a, b]].dropna().astype(int)
    if pair[a].nunique() > 1 or pair[b].nunique() > 1:
        kappas.append(cohen_kappa_score(pair[a], pair[b], weights="quadratic"))
print(f"\nStatements kept: {len(statements)}  |  all {N} raters identical: {(statements['spread'] == 0).mean():.0%}"
      f"  |  all within 1 point: {(statements['spread'] <= 1).mean():.0%}"
      f"  |  mean pairwise quadratic weighted kappa: {np.mean(kappas) if kappas else float('nan'):.2f}"
      f"  |  adjudicated: {statements['adjudicated'].sum()}")

ratings = (statements.pivot_table(index="participant_id", columns="attribute", values="rating", aggfunc="mean")
           .reindex(index=list(by_pid), columns=ATTRIBUTE_KEYS))
ratings.insert(0, "user_group", [by_pid[p]["user_group"] for p in ratings.index])
statements.to_csv(os.path.join(OUTPUT_DIR, "3_rated_statements.csv"), index=False)
ratings.to_csv(os.path.join(OUTPUT_DIR, "3_participant_ratings.csv"))

summary = ratings.groupby("user_group")[ATTRIBUTE_KEYS].agg(["mean", "std"]).round(2)
display(Markdown("**Attribute ratings by role (mean, SD)**"))
display(summary.rename(columns=SHORT))
display(ratings.rename(columns=SHORT).style.format(precision=2, na_rep="–")
        .background_gradient(cmap="Blues", vmin=1, vmax=5, subset=list(SHORT.values())))

### Optional: RoBERTa sentiment scores (as in the paper)

The CardiffNLP RoBERTa model scores every extracted statement. Scores are converted to a 1–5 scale (`3 + 2 × (P(positive) − P(negative))`), averaged per attribute, and kept as **separate clustering features**.

In [ ]:
sentiment = None
if USE_ROBERTA_SENTIMENT:
    from transformers import pipeline
    clf = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                   top_k=None, truncation=True, max_length=512)
    def to_1_5(scores):
        p = {s["label"].lower(): s["score"] for s in scores}
        return 3 + 2 * (p.get("positive", 0) - p.get("negative", 0))
    statements["sentiment"] = [to_1_5(s) for s in clf(statements["statement"].tolist(), batch_size=16)]
    sentiment = (statements.pivot_table(index="participant_id", columns="attribute", values="sentiment", aggfunc="mean")
                 .reindex(index=ratings.index, columns=ATTRIBUTE_KEYS))
    statements.to_csv(os.path.join(OUTPUT_DIR, "3_rated_statements.csv"), index=False)
    sentiment.to_csv(os.path.join(OUTPUT_DIR, "3_participant_sentiment.csv"))
    display(Markdown("**Sentiment scores by role (mean)**"))
    display(sentiment.join(ratings["user_group"]).groupby("user_group").mean().round(2).rename(columns=SHORT))

## Step 9: Agent group 4, hierarchical clustering by role (3 cluster analysts per role, 3 profile matchers)

As in the paper, each role is clustered separately: features are z-scored, distances are Euclidean, and the linkage is complete. **3 Cluster Analysts** independently judge the candidate cut points for qualitative coherence, using each member's rated statements. The cut is set by **majority vote**.

> **Researcher checkpoint:** review the dendrograms and the analysts' votes. To overrule them, set `FINAL_K` in the next cell, then run it.

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import zscore

MAX_K = 5
features = ratings[ATTRIBUTE_KEYS].add_prefix("rating:")
if sentiment is not None:
    features = features.join(sentiment.add_prefix("sentiment:"))

CLUSTER_ANALYST = f"""You are a qualitative researcher reviewing hierarchical clustering of clinicians'
adoption attribute scores (one professional role at a time). Clusters are an organizing step, not inference.
For each candidate number of clusters, judge whether the members of each cluster share coherent adoption
perceptions according to their rated statements. Consider dendrogram structure, linkage distances, cluster
sizes and qualitative coherence. Recommend the number of clusters. A single-member cluster is acceptable when
that participant's adoption pattern is clearly distinct. Research context: {RESEARCH_CONTEXT}"""

def statements_of(pid, limit=12):
    s = statements[statements["participant_id"] == pid]
    return "\n".join(f"  - {SHORT[r.attribute]} ({r.rating:.1f}): \"{r.statement}\"" for r in s.head(limit).itertuples())

cluster_data = {}
fig, axes = plt.subplots(1, len(USER_GROUPS), figsize=(7 * len(USER_GROUPS), 4.5), squeeze=False)
for ax, group in zip(axes[0], USER_GROUPS):
    F = features[ratings["user_group"] == group]
    if len(F) < 3:
        print(f"{group}: only {len(F)} participants, clustering skipped (one cluster).")
        cluster_data[group] = {"F": F, "candidates": {1: np.ones(len(F), int)}, "recommended": 1, "votes": {}}
        ax.set_visible(False)
        continue
    X = F.apply(lambda c: c.fillna(c.median())).fillna(3)          # missing evidence -> role median
    Xz = X.apply(lambda c: zscore(c) if c.std() > 0 else c * 0)
    Z = linkage(Xz.values, method="complete", metric="euclidean")
    candidates = {k: fcluster(Z, k, criterion="maxclust") for k in range(2, min(MAX_K, len(F) - 1) + 1)}
    dendrogram(Z, labels=list(F.index), ax=ax, color_threshold=0, above_threshold_color="#52514e")
    ax.set_title(f"{group}s: complete linkage, Euclidean, z-scored (n={len(F)})", loc="left", fontsize=11)
    ax.set_ylabel("Linkage distance"); ax.spines[["top", "right"]].set_visible(False)

    desc = []
    for k, lab in candidates.items():
        desc.append(f"### k = {k} (cut height range: {Z[-k, 2]:.2f}-{Z[-(k - 1), 2]:.2f})")
        for c in sorted(set(lab)):
            mem = list(F.index[lab == c])
            means = ratings.loc[mem, ATTRIBUTE_KEYS].mean().round(2).rename(SHORT).to_dict()
            desc.append(f"Cluster {c} (n={len(mem)}): {mem}\nMean ratings: {means}")
    evidence = "\n".join(f"{p}:\n{statements_of(p)}" for p in F.index)
    recs = ensemble("Cluster analyst", CLUSTER_ANALYST,
                    f"Role: {group}\nMerge heights: {np.round(Z[:, 2], 2).tolist()}\n\n" + "\n\n".join(desc)
                    + f"\n\nRated statements per participant:\n{evidence}", CutRecommendation, effort="high")
    votes = [r.recommended_k for r in recs if r.recommended_k in candidates] or [min(candidates)]
    rk, n_votes, counts = majority(votes)
    cluster_data[group] = {"F": F, "candidates": candidates, "recommended": rk, "votes": counts,
                           "recommendations": [r.model_dump() for r in recs]}
    print(f"\n{group}s: analysts voted {counts} -> k = {rk} ({n_votes} of {N} agree)")
    for i, r in enumerate(recs, 1):
        print(f"  Analyst {i} (k={r.recommended_k}): {r.rationale}")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "4_dendrograms.png"), dpi=150); plt.show()

In [ ]:
# ---------------- RESEARCHER CHECKPOINT ----------------
FINAL_K = {"Physician": None, "Nurse": None}    # None = accept the analysts' majority vote
# --------------------------------------------------------

cluster_of = {}
for group, d in cluster_data.items():
    k = FINAL_K.get(group) or d["recommended"]
    for pid, c in zip(d["F"].index, d["candidates"][k]):
        cluster_of[pid] = f"{group}-{c}"
ratings["cluster"] = ratings.index.map(cluster_of)
clusters = sorted(set(cluster_of.values()))
for c in clusters:
    print(f"{c}: {list(ratings.index[ratings['cluster'] == c])}")
display(ratings.groupby("cluster")[ATTRIBUTE_KEYS].mean().round(2).rename(columns=SHORT)
        .style.format(precision=2).background_gradient(cmap="Blues", vmin=1, vmax=5))

PROFILE_MATCHER = f"""You compare clusters from separate physician and nurse analyses. Clusters from different
roles with similar adoption patterns (attribute ratings AND underlying statements: perceived clinical value,
concerns, conditions for adoption) represent the SAME adoption profile; merge them. Keep workflow and
responsibility differences as role differences within a profile. Every cluster must appear in exactly one
profile. Give each profile a short, design-relevant name. Research context: {RESEARCH_CONTEXT}"""

PROFILE_RECONCILER = f"""{N} independent matchers proposed adoption profiles. The groupings below are FINAL: they
contain the clusters that a majority of matchers placed together. Keep every grouping exactly as given (same
clusters), and give each one a profile name, adoption pattern and role differences, drawing on the matchers'
proposals. Research context: {RESEARCH_CONTEXT}"""

cluster_brief = "\n\n".join(
    f"## {c} (n={sum(ratings['cluster'] == c)})\nMean ratings: "
    f"{ratings.loc[ratings['cluster'] == c, ATTRIBUTE_KEYS].mean().round(2).rename(SHORT).to_dict()}\n"
    + "\n".join(f"{p}:\n{statements_of(p, 8)}" for p in ratings.index[ratings["cluster"] == c])
    for c in clusters) + f"\n\nThemes:\n{theme_brief}"

print(f"\n{N} profile matchers are comparing physician and nurse clusters...")
proposals = ensemble("Profile matcher", PROFILE_MATCHER, cluster_brief, ProfileMatching, effort="high")

# Consensus: two clusters share a profile only if a majority of matchers put them together
together = Counter()
for prop in proposals:
    for prof in prop.profiles:
        members = sorted(set(c for c in prof.clusters if c in clusters))
        for a, b in combinations(members, 2):
            together[(a, b)] += 1
parent = {c: c for c in clusters}
def root(c):
    while parent[c] != c:
        c = parent[c]
    return c
for (a, b), n in together.items():
    if n > N / 2:
        parent[root(a)] = root(b)
groups = {}
for c in clusters:
    groups.setdefault(root(c), []).append(c)
groupings = list(groups.values())
print("Consensus groupings:", groupings)
print("Pair agreement:", {f"{a} + {b}": f"{n}/{N}" for (a, b), n in together.items()})

naming = run_agent(PROFILE_RECONCILER, "Final groupings:\n" + "\n".join(f"- {g}" for g in groupings)
                   + "\n\nMatchers' proposals:\n" + "\n\n".join(
                       f"## Matcher {i}\n{p.model_dump_json(indent=1)}" for i, p in enumerate(proposals, 1))
                   + f"\n\n{cluster_brief}", ProfileNaming)

prof_info, profile_of = {}, {}
for g in groupings:
    best = max(naming.profiles, key=lambda p: len(set(p.clusters) & set(g)), default=None)
    name = best.profile_name if best and set(best.clusters) & set(g) and best.profile_name not in prof_info \
        else f"Profile {' + '.join(g)}"
    prof_info[name] = Profile(profile_name=name, clusters=g,
                              adoption_pattern=best.adoption_pattern if best else "",
                              role_differences=best.role_differences if best else "")
    for c in g:
        profile_of[c] = name
ratings["profile"] = ratings["cluster"].map(profile_of)
profiles = list(dict.fromkeys(ratings["profile"]))
print(pd.crosstab(ratings["profile"], ratings["user_group"], margins=True))
save_json("4_clusters_and_profiles.json", {
    "final_k": {g: FINAL_K.get(g) or d["recommended"] for g, d in cluster_data.items()},
    "analyst_votes": {g: d["votes"] for g, d in cluster_data.items()},
    "analyst_recommendations": {g: d.get("recommendations", []) for g, d in cluster_data.items()},
    "cluster_of": cluster_of, "matcher_proposals": [p.model_dump() for p in proposals],
    "pair_agreement": {f"{a} + {b}": n for (a, b), n in together.items()},
    "profiles": {n: p.model_dump() for n, p in prof_info.items()}})
ratings.to_csv(os.path.join(OUTPUT_DIR, "4_participants_with_profiles.csv"))

## Step 10: Agent group 5, persona cards (1 designer + 3 verifiers per profile)

In [ ]:
import html as _html

PERSONA_DESIGNER = f"""You design evidence-based CLINICIAN ADOPTION PERSONAS. Write one persona for the given
adoption profile, grounded in its members' rated statements, attribute ratings and the themes. Quotes must
be copied verbatim from participants (never the interviewer). Describe how this profile evaluates the system,
what drives and blocks adoption, and the conditions for use. Give role-specific needs for each role present.
Design requirements must be concrete (dashboard content, presentation, explanations). Implementation
strategies must match the profile's readiness (training, evidence, trial use, peer support).
Research context: {RESEARCH_CONTEXT}"""

PERSONA_VERIFIERS = {
    "Evidence verifier": "Check that every claim, need and quote in the persona is traceable to the members' "
                         "transcripts and rated statements, and that quotes are verbatim participant words.",
    "Profile-consistency verifier": "Check that the persona is consistent with the profile's attribute ratings and "
                                    "adoption pattern, and that role-specific needs reflect the physician / nurse "
                                    "differences in the data.",
    "Bias and actionability verifier": "Check for stereotypes or assumptions (gender, age, role) not supported by the "
                                       "data, and that design requirements and implementation strategies are concrete "
                                       "and match the profile's adoption readiness.",
}

def design_persona(name):
    mem = list(ratings.index[ratings["profile"] == name])
    p = prof_info.get(name)
    member_list = ", ".join(f"{m} ({by_pid[m]['user_group']})" for m in mem)
    task = (f"Adoption profile: {name}\nPattern: {p.adoption_pattern if p else ''}\n"
            f"Role differences: {p.role_differences if p else ''}\n"
            f"Members: {member_list}\n"
            f"Mean ratings: {ratings.loc[mem, ATTRIBUTE_KEYS].mean().round(2).rename(SHORT).to_dict()}\n\n"
            f"Themes:\n{theme_brief}\n\nRated statements:\n" + "\n".join(f"{m}:\n{statements_of(m, 30)}" for m in mem)
            + "\n\nMember transcripts:\n\n"
            + "\n\n".join(f"### {m} ({by_pid[m]['user_group']})\n{by_pid[m]['clean_text']}" for m in mem))
    card = run_agent(PERSONA_DESIGNER, task, PersonaCard, effort="high")
    checks = parallel(lambda kv: (kv[0], run_agent(
        f"You are the {kv[0]} for research personas, working independently of two other verifiers. {kv[1]} "
        f"Approve only if there are no substantive issues.", f"{task}\n\nPERSONA TO VERIFY:\n{card.model_dump_json(indent=1)}",
        PersonaVerification)), PERSONA_VERIFIERS.items())
    issues = [f"{who}: {i}" for who, v in checks if not v.approved for i in v.issues]
    revised = False
    if issues:
        card = run_agent(PERSONA_DESIGNER, f"{task}\n\nYOUR DRAFT:\n{card.model_dump_json(indent=1)}\n\n"
                         "Independent verifiers raised these issues. Fix every one and return the full persona:\n- "
                         + "\n- ".join(issues), PersonaCard, effort="high")
        revised = True
    persona_checks[name] = {"approved_by": sum(v.approved for _, v in checks), "revised": revised,
                            "issues": issues}
    print(f"  {name}: {persona_checks[name]['approved_by']}/{len(checks)} verifiers approved the draft"
          + (", revised" if revised else ""))
    return card

persona_checks = {}
print("Group 5: persona designers and verifiers are working...")
cards = dict(zip(profiles, parallel(design_persona, profiles)))

def check_line(name):
    c = persona_checks.get(name, {})
    if not c:
        return ""
    return (f'<span class="meta">Verified by {len(PERSONA_VERIFIERS)} agents: {c["approved_by"]} approved the draft'
            f'{"; revised to address their issues" if c["revised"] else ""}</span>')


SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
def _li(items): return "".join(f"<li>{_html.escape(i)}</li>" for i in items)
def _q(quote, mem):
    ok = any(verbatim(quote, by_pid[m]["raw_text"]) for m in mem)
    return (f'<blockquote>“{_html.escape(quote.strip(chr(34) + "“”"))}”'
            f'<span class="chk">{"✓ verified in transcript" if ok else "⚠ not found in transcripts"}</span></blockquote>')

def card_html(i, name, card):
    mem = list(ratings.index[ratings["profile"] == name])
    color = SERIES[i % len(SERIES)]
    comp = ", ".join(f"{n} {g.lower()}{'s' if n > 1 else ''}"
                     for g, n in ratings.loc[mem, "user_group"].value_counts().items())
    bars = ""
    for k in ATTRIBUTE_KEYS:
        v = ratings.loc[mem, k].mean()
        width, label = (0, "–") if pd.isna(v) else (v / 5 * 100, f"{v:.2f}")
        bars += (f'<div class="bar"><span>{_html.escape(SHORT[k])}</span><div class="track">'
                 f'<div class="fill" style="width:{width:.0f}%"></div></div><b>{label}</b></div>')
    roles = "".join(f"<div><h4>{_html.escape(r.role)}</h4><ul>{_li(r.needs)}</ul></div>" for r in card.role_specific_needs)
    words = [w for w in card.persona_name.split() if w.rstrip(".").lower() not in ("dr", "nurse", "rn", "md", "np")]
    initials = "".join(w[0] for w in words[:2]).upper()
    return f"""
<div class="card">
  <div class="head" style="border-top:6px solid {color}">
    <div class="avatar" style="background:{color}">{_html.escape(initials)}</div>
    <div><div class="profile">{_html.escape(name)}</div><h2>{_html.escape(card.persona_name)}</h2>
    <span class="meta">{len(mem)} of {len(ratings)} clinicians · {comp}</span>{check_line(name)}</div>
  </div>
  {_q(card.tagline_quote, mem)}
  <p>{_html.escape(card.description)}</p>
  <h3>Adoption attributes <small>(mean qualitative rating, 1–5)</small></h3>{bars}
  <div class="cols">
    <div><h3>Goals</h3><ul>{_li(card.goals)}</ul></div>
    <div><h3>Conditions for use</h3><ul>{_li(card.conditions_for_use)}</ul></div>
    <div><h3>Adoption drivers</h3><ul>{_li(card.adoption_drivers)}</ul></div>
    <div><h3>Adoption concerns</h3><ul>{_li(card.adoption_concerns)}</ul></div>
  </div>
  <h3>Role-specific needs</h3><div class="cols">{roles}</div>
  <h3>Design requirements</h3><ul>{_li(card.design_requirements)}</ul>
  <h3>Implementation strategies</h3><ul>{_li(card.implementation_strategies)}</ul>
  <h3>In their words</h3>{"".join(_q(q, mem) for q in card.supporting_quotes)}
</div>"""

CSS = """
:root{--bg:#f4f4f2;--card:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--line:#e4e3df;--track:#e9e8e4;--fill:#2a78d6}
@media (prefers-color-scheme: dark){:root{--bg:#111110;--card:#1a1a19;--ink:#f0efec;--ink2:#c3c2b7;--line:#383835;--track:#2c2c2a;--fill:#3987e5}}
body{background:var(--bg);margin:0}
.wrap{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:var(--ink);display:grid;
      grid-template-columns:repeat(auto-fill,minmax(380px,1fr));gap:20px;padding:16px;background:var(--bg)}
.card{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:20px;font-size:14px;line-height:1.45}
.head{display:flex;gap:14px;align-items:center;margin:-20px -20px 8px;padding:16px 20px 0;border-radius:14px 14px 0 0}
.avatar{width:56px;height:56px;border-radius:50%;color:#fff;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:20px;flex:none}
.profile{font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--ink2)}
h2{margin:0;font-size:20px} h3{font-size:12px;text-transform:uppercase;letter-spacing:.06em;color:var(--ink2);margin:16px 0 6px}
h3 small{text-transform:none;letter-spacing:0;font-weight:400} h4{margin:4px 0;font-size:13px}
.meta{display:block;font-size:12px;color:var(--ink2);margin-top:4px}
blockquote{margin:10px 0;padding:8px 12px;border-left:3px solid var(--line);font-style:italic}
.chk{display:block;font-style:normal;font-size:11px;color:var(--ink2);margin-top:2px}
.bar{display:grid;grid-template-columns:160px 1fr 36px;gap:8px;align-items:center;font-size:12px;color:var(--ink2);margin:3px 0}
.track{height:8px;background:var(--track);border-radius:4px}.fill{height:8px;background:var(--fill);border-radius:4px}
.bar b{color:var(--ink);text-align:right}
.cols{display:grid;grid-template-columns:1fr 1fr;gap:0 16px} ul{margin:0;padding-left:18px}
@media (max-width:480px){.cols{grid-template-columns:1fr}.wrap{grid-template-columns:1fr}}
"""
cards_html = f'<style>{CSS}</style><div class="wrap">' + "".join(
    card_html(i, n, cards[n]) for i, n in enumerate(profiles)) + "</div>"
from IPython.display import HTML
display(HTML(cards_html))
with open(os.path.join(OUTPUT_DIR, "5_persona_cards.html"), "w", encoding="utf-8") as f:
    f.write('<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,'
            f'initial-scale=1"><title>Clinician Adoption Personas</title></head><body>{cards_html}</body></html>')
save_json("5_persona_cards.json", {n: c.model_dump() for n, c in cards.items()})

## Step 11: Living personas, talk to them

Each persona is an agent that answers **in character**, grounded only in its own cluster's data: the members' rated statements, attribute scores, themes and transcripts.

* Ask about needs, frustrations, workflow, trust, what would make them adopt the system, or how they would react to a design idea.
* Choose **"All personas (panel)"** to put the same question to every persona and compare their answers side by side.
* Every answer is reviewed by **2 checker agents** (grounding and character), so 3 agents are involved in each answer. Set `VERIFY_REPLIES = False` for faster chats.
* Each answer shows a **grounding label** (*grounded in the data*, *partly inferred* or *not covered by the data*) and the participant quotes behind it. Quotes are checked against the transcripts.

**Run this step on its own later:** run Steps 1–2, then this step. If the pipeline isn't in memory, the cell asks you to upload `persona_dossiers.json`, which every pipeline run saves to `persona_outputs`.

In [ ]:
import html as _html
from pydantic import BaseModel, Field
from typing import Literal
if "verbatim" not in globals():           # running this step on its own (Steps 1-2 must have run)
    def verbatim(quote, text):
        norm = lambda s: re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", s.lower())).strip()
        q = norm(quote.strip('"“”'))
        return bool(q) and q in norm(text)
INCLUDE_TRANSCRIPTS = True      # give each persona its members' full transcripts for richer answers
PERSONA_EFFORT = "medium"       # "low" for faster replies

DOSSIER_FILE = os.path.join(OUTPUT_DIR, "persona_dossiers.json")

def build_dossiers():
    out = {}
    for name in profiles:
        mem = list(ratings.index[ratings["profile"] == name])
        st = statements[statements["participant_id"].isin(mem)]
        by_role = ratings.loc[mem].groupby("user_group")[ATTRIBUTE_KEYS].mean().round(2)
        out[name] = {
            "profile_name": name,
            "persona": cards[name].model_dump(),
            "members": {m: by_pid[m]["user_group"] for m in mem},
            "attribute_definitions": {v["name"]: v["definition"] for v in ATTRIBUTES.values()},
            "mean_attribute_ratings_1_to_5": {SHORT[k]: round(float(v), 2)
                                              for k, v in ratings.loc[mem, ATTRIBUTE_KEYS].mean().items() if pd.notna(v)},
            "ratings_by_role": {g: {SHORT[k]: v for k, v in row.dropna().items()} for g, row in by_role.iterrows()},
            "rated_statements": [{"participant_id": r.participant_id, "role": r.user_group, "attribute": SHORT[r.attribute],
                                  "rating": round(float(r.rating), 2), "statement": r.statement} for r in st.itertuples()],
            "themes": [th.model_dump() for th in thematic_map.themes],
            "transcripts": {m: by_pid[m]["clean_text"] for m in mem} if INCLUDE_TRANSCRIPTS else {},
        }
    return out

if "cards" in globals() and "statements" in globals():
    dossiers = build_dossiers()
    save_json("persona_dossiers.json", dossiers)
else:
    from google.colab import files
    print("Upload persona_dossiers.json from a previous run:")
    dossiers = json.loads(next(iter(files.upload().values())).decode("utf-8"))
print("Personas ready:", ", ".join(f"{d['persona']['persona_name']} ({n})" for n, d in dossiers.items()))

In [ ]:
class Evidence(BaseModel):
    participant_id: str
    quote: str = Field(description="Copied verbatim from the rated statements or transcripts")

class PersonaReply(BaseModel):
    reply: str = Field(description="The persona's answer, in first person, in character")
    grounding: Literal["grounded in the data", "partly inferred", "not covered by the data"]
    evidence: list[Evidence]

def persona_system_prompt(d):
    p = d["persona"]
    comp = ", ".join(f"{list(d['members'].values()).count(r)} {r.lower()}(s)" for r in dict.fromkeys(d["members"].values()))
    return f"""You are {p['persona_name']}, a clinician adoption persona representing the "{d['profile_name']}" profile.
You are a composite of {len(d['members'])} real clinicians ({comp}) who evaluated an AI-enabled clinical decision
support system for chronic pain assessment that uses eye-movement measures. A researcher or designer is now
interviewing you.

How to behave:
- Speak in the first person as this clinician, naturally and conversationally, like a real interviewee.
- Everything you say about your needs, frustrations, trust, workflow, conditions for use and opinions must come from
  the research data below. Your adoption attribute ratings describe your overall stance; stay consistent with them.
- When members of your profile differ (including physician vs nurse differences), say so naturally,
  e.g. "When I'm working as a nurse..." or "Some of us feel...".
- If you are asked about something the data does not cover, say it did not come up in the research. You may
  offer a cautious guess about how you would react, but label it clearly as a guess, and set grounding accordingly.
- Never invent experiences, numbers or quotes. Evidence quotes must be copied verbatim from the data, with the
  participant ID.
- If the user steps out of the role-play and asks about the data itself (e.g. "which participants said that?"),
  answer plainly as a research assistant.

Research context: {RESEARCH_CONTEXT}

RESEARCH DATA FOR THIS PERSONA:
{json.dumps(d, ensure_ascii=False, indent=1)}"""

chat_history = {name: [] for name in dossiers}
check_log = {name: [] for name in dossiers}

def call_persona(system, messages):
    """One persona turn. Returns the parsed reply and the assistant content to append to the history."""
    with client.beta.messages.stream(
        model=MODEL, max_tokens=16000,
        system=[{"type": "text", "text": system, "cache_control": {"type": "ephemeral"}}],  # cache the dossier
        messages=messages, output_config={"effort": PERSONA_EFFORT}, output_format=PersonaReply,
        betas=["server-side-fallback-2026-07-01"], fallbacks="default",
    ) as stream:
        msg = stream.get_final_message()
    if msg.stop_reason == "refusal":
        raise RuntimeError(f"Request declined: {msg.stop_details}")
    content = [{k: v for k, v in b.model_dump(exclude_none=True).items() if k != "parsed_output"} for b in msg.content]
    return msg.parsed_output, content

VERIFY_REPLIES = True    # 2 checker agents review every persona answer (3 agents per answer in total)

REPLY_CHECKERS = {
    "Grounding checker": "Check that every claim in the persona's answer is supported by the research data, that "
                         "anything not in the data is clearly labelled as a guess, and that the grounding label is honest.",
    "Character checker": "Check that the answer stays consistent with this persona's adoption profile, attribute ratings "
                         "and earlier answers, reflects disagreements and physician / nurse differences where relevant, "
                         "and does not invent experiences, numbers or quotes.",
}

def check_reply(d, question, reply):
    data = json.dumps({k: v for k, v in d.items() if k != "transcripts"}, ensure_ascii=False)
    return parallel(lambda kv: (kv[0], run_agent(
        f"You are the {kv[0]} for a research persona, working independently of another checker. {kv[1]}",
        f"RESEARCH DATA:\n{data}\n\nQUESTION: {question}\n\nPERSONA ANSWER:\n{reply.model_dump_json(indent=1)}",
        ReplyCheck, effort="low")), REPLY_CHECKERS.items())

def ask(persona: str, question: str):
    """Ask one persona a question; returns (reply, checks). The conversation history is kept per persona."""
    d = dossiers[persona]
    history = chat_history[persona]
    history.append({"role": "user", "content": question})
    try:
        reply, content = call_persona(persona_system_prompt(d), history)
    except Exception:
        history.pop()                     # keep the history valid if the call failed
        raise
    history.append({"role": "assistant", "content": content})
    checks = check_reply(d, question, reply) if VERIFY_REPLIES else []
    check_log[persona].append({"question": question, "checks": {who: c.model_dump() for who, c in checks}})
    return reply, checks

def verified_in(d, quote):
    return any(verbatim(quote, text) for text in d["transcripts"].values()) or \
           any(verbatim(quote, s["statement"]) for s in d["rated_statements"])

GROUNDING_STYLE = {"grounded in the data": "✓ grounded in the data", "partly inferred": "◐ partly inferred",
                   "not covered by the data": "○ not covered by the data (guess)"}

CHECK_ICON = {"pass": "✓", "minor issues": "◐", "fail": "✗"}

def reply_html(persona, reply, checks=()):
    d = dossiers[persona]
    p = d["persona"]
    ev = "".join(
        f'<li><b>{_html.escape(e.participant_id)}</b>: “{_html.escape(e.quote)}” '
        f'<span class="lp-chk">{"✓" if verified_in(d, e.quote) else "⚠ not found"}</span></li>' for e in reply.evidence)
    return f"""<div class="lp-msg lp-bot"><div class="lp-who">{_html.escape(p['persona_name'])} · {_html.escape(persona)}</div>
      <div>{_html.escape(reply.reply).replace(chr(10), "<br>")}</div>
      <details><summary>{GROUNDING_STYLE.get(reply.grounding, reply.grounding)} · {len(reply.evidence)} quotes</summary>
      <ul>{ev or "<li>No supporting quotes.</li>"}</ul></details>{checks_html(checks)}</div>"""

def checks_html(checks):
    if not checks:
        return ""
    summary = " · ".join(f"{CHECK_ICON.get(c.verdict, '?')} {who}: {c.verdict}" for who, c in checks)
    issues = "".join(f"<li><b>{_html.escape(who)}</b>: {_html.escape(i)}</li>" for who, c in checks for i in c.issues)
    return f"<details><summary>{summary}</summary><ul>{issues or '<li>No issues found.</li>'}</ul></details>"

def user_html(text):
    return f'<div class="lp-msg lp-user"><div class="lp-who">You</div><div>{_html.escape(text)}</div></div>'

CHAT_CSS = """<style>
.lp-msg{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;font-size:14px;line-height:1.5;max-width:760px;
        padding:10px 14px;border-radius:12px;margin:8px 0;border:1px solid #e4e3df}
.lp-bot{background:#fcfcfb;color:#0b0b0b}.lp-user{background:#e8f1fc;color:#0b0b0b;margin-left:auto;max-width:600px}
.lp-who{font-size:12px;font-weight:600;color:#52514e;margin-bottom:2px}
.lp-msg details{margin-top:6px;font-size:12px;color:#52514e}.lp-msg ul{margin:4px 0;padding-left:18px}
.lp-chk{font-size:11px;color:#52514e}
</style>"""

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

PANEL = "All personas (panel)"
who = widgets.Dropdown(options=list(dossiers) + [PANEL], description="Talk to:", layout=widgets.Layout(width="420px"))
box = widgets.Textarea(placeholder="e.g. What frustrates you about the dashboard? What would make you use it every day?",
                       layout=widgets.Layout(width="760px", height="70px"))
send = widgets.Button(description="Send", button_style="primary")
reset = widgets.Button(description="New conversation")
log = widgets.Output(layout=widgets.Layout(max_height="600px", overflow="auto"))

def on_send(_):
    q = box.value.strip()
    if not q:
        return
    box.value = ""
    send.disabled = True
    with log:
        display(HTML(CHAT_CSS + user_html(q)))
        targets = list(dossiers) if who.value == PANEL else [who.value]
        try:
            results = parallel(lambda name: ask(name, q), targets)
            for name, (r, checks) in zip(targets, results):
                display(HTML(reply_html(name, r, checks)))
        except Exception as e:
            display(HTML(f"<b>Error:</b> {_html.escape(str(e))}"))
    send.disabled = False

def on_reset(_):
    for h in chat_history.values():
        h.clear()
    log.clear_output()

send.on_click(on_send)
reset.on_click(on_reset)
display(HTML(CHAT_CSS), who, box, widgets.HBox([send, reset]), log)

# No widgets? Use ask() directly, e.g.:
# name = list(dossiers)[0]
# display(HTML(CHAT_CSS + reply_html(name, *ask(name, "What worries you most about this system?"))))

In [ ]:
# Save all persona conversations (e.g. to include persona interviews in design documentation)
def export_conversations():
    out = {}
    for name, h in chat_history.items():
        turns = []
        for m in h:
            if m["role"] == "user":
                turns.append({"researcher": m["content"]})
            else:
                text = "".join(b.get("text", "") for b in m["content"] if b.get("type") == "text")
                turns.append({"persona": json.loads(text) if text.strip().startswith("{") else text})
        if turns:
            out[name] = turns
    save_json("persona_conversations.json", {"conversations": out, "answer_checks": check_log})
    print(f"Saved {sum(len(v) for v in out.values())} turns to {OUTPUT_DIR}/persona_conversations.json")
export_conversations()

## Step 12 (optional): Post hoc correlations with usability measures

Upload a CSV with columns `participant_id`, `SUS`, `PEOU` (for example, the SUS score and perceived ease of use on a 1–7 scale). The notebook computes Pearson correlations with the attribute ratings, as in the paper's post hoc analysis.

In [ ]:
RUN_POSTHOC = False

if RUN_POSTHOC:
    from scipy.stats import pearsonr
    from matplotlib.colors import LinearSegmentedColormap
    from google.colab import files
    import io
    up = files.upload()
    usab = pd.read_csv(io.BytesIO(next(iter(up.values())))).set_index("participant_id")
    data = ratings[ATTRIBUTE_KEYS].rename(columns=SHORT).join(usab[["SUS", "PEOU"]], how="inner")
    cols = list(data.columns)
    r = data.corr(method="pearson")
    p = pd.DataFrame([[pearsonr(*data[[a, b]].dropna().T.values)[1] if a != b else 0 for b in cols] for a in cols],
                     index=cols, columns=cols)
    diverging = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
    fig, ax = plt.subplots(figsize=(8, 6.5))
    im = ax.imshow(r.values, cmap=diverging, vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)), cols, rotation=40, ha="right"); ax.set_yticks(range(len(cols)), cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            star = "*" if i != j and p.iloc[i, j] < .05 else ""
            ax.text(j, i, f"{r.iloc[i, j]:.2f}{star}", ha="center", va="center", fontsize=9, color="#0b0b0b")
    fig.colorbar(im, ax=ax, label="Pearson r"); ax.set_title(f"Correlations (N={len(data)}; * P<.05)", loc="left")
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "6_posthoc_correlations.png"), dpi=150); plt.show()
    r.round(2).to_csv(os.path.join(OUTPUT_DIR, "6_posthoc_correlations.csv"))

## Step 13: Download everything

In [ ]:
import shutil
shutil.make_archive("persona_outputs", "zip", OUTPUT_DIR)
print("Files:", sorted(os.listdir(OUTPUT_DIR)))
try:
    from google.colab import files
    files.download("persona_outputs.zip")
except ImportError:
    print("Saved persona_outputs.zip")